<a href="https://colab.research.google.com/github/Harshu120/UE25CS645BC2_PES1PG25CS102_Fashion_MNIST_CNN1/blob/main/UE25CS645BC2_PES1PG25CS102.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from tensorflow.keras.datasets import fashion_mnist

In [ ]:
import numpy as np
from tensorflow.keras.datasets import fashion_mnist

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
import numpy as np
from tensorflow.keras.datasets import fashion_mnist

(train_x, train_y), (test_x, test_y) = fashion_mnist.load_data()

train_x = train_x / 255.0
test_x = test_x / 255.0

print(train_x.shape)
print(test_x.shape)

(60000, 28, 28)
(10000, 28, 28)


In [ ]:
class ConvLayer:

    def __init__(self, filters_count, kernel_size):

        self.filters_count = filters_count
        self.kernel_size = kernel_size

        self.filters = np.random.randn(
            filters_count,
            kernel_size,
            kernel_size
        ) / (kernel_size * kernel_size)

    def forward(self, image):

        self.image = image

        height, width = image.shape

        result = np.zeros((
            self.filters_count,
            height - self.kernel_size + 1,
            width - self.kernel_size + 1
        ))

        for filt in range(self.filters_count):

            for row in range(height - self.kernel_size + 1):

                for col in range(width - self.kernel_size + 1):

                    patch = image[
                        row:row+self.kernel_size,
                        col:col+self.kernel_size
                    ]

                    result[filt, row, col] = np.sum(
                        patch * self.filters[filt]
                    )

        return result

    def backward(self, grad_output, lr):

        grad_filters = np.zeros(self.filters.shape)

        for filt in range(self.filters_count):

            for row in range(self.image.shape[0] - self.kernel_size + 1):

                for col in range(self.image.shape[1] - self.kernel_size + 1):

                    patch = self.image[
                        row:row+self.kernel_size,
                        col:col+self.kernel_size
                    ]

                    grad_filters[filt] += (
                        grad_output[filt, row, col] * patch
                    )

        self.filters -= lr * grad_filters

In [ ]:
class MaxPool:

    def __init__(self, pool_size):

        self.pool_size = pool_size

    def forward(self, feature_map):

        self.feature_map = feature_map

        total_filters, height, width = feature_map.shape

        pooled = np.zeros((
            total_filters,
            height // 2,
            width // 2
        ))

        for filt in range(total_filters):

            for row in range(height // 2):

                for col in range(width // 2):

                    section = feature_map[
                        filt,
                        row*2:row*2+2,
                        col*2:col*2+2
                    ]

In [ ]:
def flatten(values):

    return values.flatten()

In [ ]:
class Dense:

    def __init__(self, input_nodes, output_nodes):

        self.weights = np.random.randn(
            input_nodes,
            output_nodes
        ) / input_nodes

        self.bias = np.zeros(output_nodes)

    def forward(self, values):

        self.values = values

        return np.dot(values, self.weights) + self.bias

    def backward(self, gradient_output, lr):

        gradient_weights = np.outer(
            self.values,
            gradient_output
        )

        gradient_input = np.dot(
            self.weights,
            gradient_output
        )

        self.weights -= lr * gradient_weights

        self.bias -= lr * gradient_output

        return gradient_input

In [ ]:
def softmax(values):

    exponent = np.exp(values - np.max(values))

    return exponent / np.sum(exponent)

In [ ]:
def cross_entropy(probabilities, target_label):

    return -np.log(probabilities[target_label])


In [ ]:
conv = ConvLayer(filters_count=8, kernel_size=3)

dense = Dense(26 * 26 * 8, 10)

In [ ]:
def forward(image, label):

    result = conv.forward(image)

    result = flatten(result)

    result = dense.forward(result)

    probabilities = softmax(result)

    loss = cross_entropy(probabilities, label)

    return probabilities, loss

In [ ]:
def train(image, label, lr=0.005):

    # forward pass
    probabilities, loss = forward(image, label)

    # gradient calculation
    gradient = probabilities.copy()

    gradient[label] -= 1

    # dense backward
    backprop = dense.backward(
        gradient,
        lr
    )

    # reshape output
    backprop = backprop.reshape(8, 26, 26)

    # convolution backward
    conv.backward(
        backprop,
        lr
            )

    return loss

In [ ]:
print("\nTraining Started...\n")

for epoch in range(3):

    print("Epoch:", epoch + 1)

    epoch_loss = 0

    for step in range(10000):

        image = train_x[step]

        label = train_y[step]

        loss = train(image, label)

        epoch_loss += loss

        if step % 1000 == 0:

            print(
                "Step:",
                step,
                "Loss:",
                round(loss, 3)
            )

    print("Average Loss:", epoch_loss / 1000)

    print()


Training Started...

Epoch: 1
Step: 0 Loss: 2.301
Step: 1000 Loss: 0.558
Step: 2000 Loss: 0.455
Step: 3000 Loss: 0.163
Step: 4000 Loss: 0.039
Step: 5000 Loss: 0.104
Step: 6000 Loss: 0.028
Step: 7000 Loss: 0.01
Step: 8000 Loss: 0.394
Step: 9000 Loss: 0.023
Average Loss: 7.042412241069571

Epoch: 2
Step: 0 Loss: 0.026
Step: 1000 Loss: 0.071
Step: 2000 Loss: 0.073
Step: 3000 Loss: 0.019
Step: 4000 Loss: 0.036
Step: 5000 Loss: 0.099
Step: 6000 Loss: 0.022
Step: 7000 Loss: 0.003
Step: 8000 Loss: 0.454
Step: 9000 Loss: 0.021
Average Loss: 5.143527723014511

Epoch: 3
Step: 0 Loss: 0.01
Step: 1000 Loss: 0.077
Step: 2000 Loss: 0.079
Step: 3000 Loss: 0.01
Step: 4000 Loss: 0.026
Step: 5000 Loss: 0.107
Step: 6000 Loss: 0.031
Step: 7000 Loss: 0.001
Step: 8000 Loss: 0.49
Step: 9000 Loss: 0.02
Average Loss: 4.664415426961179



In [ ]:
correct_predictions = 0

for index in range(1000):

    image = test_x[index]

    label = test_y[index]

    probabilities, loss = forward(image, label)

    predicted_label = np.argmax(probabilities)

    if predicted_label == label:

        correct_predictions += 1

accuracy = (correct_predictions / 1000) * 100

print("\nFinal Accuracy: %.2f%%" % accuracy)


Final Accuracy: 81.20%
